In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "dbw_retail_lakehouse_dev_eas_001"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

LAKE_ROOT = "abfss://lakehouse@stretaildeveas001.dfs.core.windows.net"

In [0]:
incremental_customers = [
    (
        "C007",
        "Patricia",
        "Lopez",
        "patricia.new@example.com",
        "PH",
        "2026-09-05T12:15:00"
    ),
    (
        "C008",
        "Miguel",
        "Garcia",
        "miguel.garcia@example.com",
        "PH",
        "2026-09-08T10:00:00"
    ),
    (
        "C009",
        "Sophia",
        "Tan",
        "sophia.tan@example.com",
        "SG",
        "2026-09-08T10:30:00"
    )
]

columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "country",
    "created_at"
]

df_incremental_customers = spark.createDataFrame(
    incremental_customers,
    columns
)

display(df_incremental_customers)

In [0]:
df_incremental_customers_bronze = (
    df_incremental_customers
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.lit("simulated_incremental_batch_2026-09-08.csv")
    )
)

In [0]:
display(df_incremental_customers_bronze)

In [0]:
print(
    "before:",
    spark.table(f"{BRONZE}.customers").count()
)

In [0]:
bronze_customers_delta = DeltaTable.forName(
    spark,
    f"{BRONZE}.customers"
)

In [0]:
(
    bronze_customers_delta.alias("target")
    .merge(
        df_incremental_customers_bronze.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(
    spark.table(f"{BRONZE}.customers")
    .orderBy("customer_id")
)

In [0]:
print(
    "after:",
    spark.table(f"{BRONZE}.customers").count()
)

In [0]:
display(
    spark.table(f"{BRONZE}.customers")
    .filter(F.col("customer_id") == "C007")
)

In [0]:
spark.table(f"{BRONZE}.customers").count()

In [0]:
df_customers_bronze = spark.table(
    f"{BRONZE}.customers"
)

In [0]:
df_customers_incremental_silver = (
    df_customers_bronze
    .withColumn(
        "customer_id",
        F.trim("customer_id")
    )
    .withColumn(
        "first_name",
        F.initcap(F.trim("first_name"))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim("last_name"))
    )
    .withColumn(
        "email",
        F.lower(F.trim("email"))
    )
    .withColumn(
        "country",
        F.upper(F.trim("country"))
    )
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("customer_id") != "")
        & F.col("email").isNotNull()
        & (F.col("email") != "")
        & F.col("created_at").isNotNull()
    )
)

In [0]:
silver_customers_delta = DeltaTable.forName(
    spark,
    f"{SILVER}.customers"
)

(
    silver_customers_delta.alias("target")
    .merge(
        df_customers_incremental_silver.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(
    spark.table(f"{SILVER}.customers")
    .orderBy("customer_id")
)

In [0]:
%sql

DESCRIBE HISTORY
dbw_retail_lakehouse_dev_eas_001.bronze.customers;

In [0]:
%sql

SELECT COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.bronze.customers;

In [0]:
%sql

SELECT *
FROM dbw_retail_lakehouse_dev_eas_001.bronze.customers
VERSION AS OF 1;